# Chapter 17 &mdash; Canonicity via Myhill&ndash;Nerode, Hash Consing, and Apply

**Concept 6 of the Chapter 17 decomposition:** *Canonicity via Myhill–Nerode, Hash Consing, and the Apply Operation*

BDDs for a function are isomorphic given a variable order, so equality checking is a pointer comparison.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter17/Concept-Canonicity-And-Apply/Concept-Canonicity-And-Apply.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The payoff of the whole construction.

**Canonicity.** For a fixed variable order, two reduced BDDs represent the same
function **iff they are isomorphic**. This is Myhill&ndash;Nerode (Chapter 6, Concept 7)
transplanted: the minimal DFA of a language is unique, so the reduced BDD of a
function is too.

**Hash consing** turns isomorphism into **pointer equality**. Because `mk` never
creates a duplicate, two structurally identical BDDs are the *same object*, and
equivalence checking is $O(1)$ rather than a graph traversal.

**Apply** combines two BDDs under a binary operation by recursing on the top variable
and memoising on the pair of nodes. With the cache it runs in $O(|f|\cdot|g|)$ &mdash;
which is why BDD packages can do real work.

Together: tautology checking, equivalence checking and satisfiability all become
trivial *given* the BDD.

## 2. Definitions

### The BDD package

In [ ]:
# --- a minimal BDD package ----------------------------------------------
# A node is either the terminal 0/1, or ('n', var_index, low, high) where
# low is the 0-branch and high the 1-branch.  Hash consing (the `unique`
# table) is what makes the representation canonical: structurally equal
# subgraphs become the SAME Python object, so equality is pointer equality.
ZERO, ONE = 0, 1

class BDD:
    def __init__(self, nvars):
        self.nvars = nvars
        self.unique = {}          # (var, low, high) -> node  -- hash consing
        self.apply_cache = {}

    def mk(self, var, low, high):
        if low is high: return low            # REDUCTION 1: skip a useless test
        key = (var, id(low), id(high), self._k(low), self._k(high))
        if key in self.unique: return self.unique[key]   # REDUCTION 2: share
        node = ('n', var, low, high)
        self.unique[key] = node
        return node

    def _k(self, n):
        return n if n in (ZERO, ONE) else ('n', n[1], self._k(n[2]), self._k(n[3]))

    def var(self, i):
        return self.mk(i, ZERO, ONE)

    def apply(self, op, a, b):
        key = (op, self._k(a), self._k(b))
        if key in self.apply_cache: return self.apply_cache[key]
        if a in (ZERO, ONE) and b in (ZERO, ONE):
            r = ONE if op(bool(a), bool(b)) else ZERO
        else:
            va = a[1] if a not in (ZERO, ONE) else self.nvars
            vb = b[1] if b not in (ZERO, ONE) else self.nvars
            v = min(va, vb)
            al, ah = (a[2], a[3]) if va == v else (a, a)
            bl, bh = (b[2], b[3]) if vb == v else (b, b)
            r = self.mk(v, self.apply(op, al, bl), self.apply(op, ah, bh))
        self.apply_cache[key] = r
        return r

    def NOT(self, a):  return self.apply(lambda x, y: not x, a, a)
    def AND(self, a, b): return self.apply(lambda x, y: x and y, a, b)
    def OR(self, a, b):  return self.apply(lambda x, y: x or y, a, b)
    def XOR(self, a, b): return self.apply(lambda x, y: x != y, a, b)

    def evaluate(self, node, assign):
        while node not in (ZERO, ONE):
            node = node[3] if assign[node[1]] else node[2]
        return bool(node)

    def size(self, node):
        seen = set()
        def walk(n):
            if n in (ZERO, ONE): return
            k = self._k(n)
            if k in seen: return
            seen.add(k); walk(n[2]); walk(n[3])
        walk(node)
        return len(seen)

    def onset(self, node, order=None):
        from itertools import product
        out = []
        for bits in product([False, True], repeat=self.nvars):
            a = {i: bits[i] for i in range(self.nvars)}
            if self.evaluate(node, a):
                out.append(''.join('1' if bits[i] else '0' for i in range(self.nvars)))
        return sorted(out)

### Deciding things, once you have the BDD

In [ ]:
def is_tautology(b, g): return g is ONE
def is_unsat(b, g):     return g is ZERO
def equivalent(b, f, g): return f is g          # pointer equality!

## 3. Tests

**Canonicity:** two different expressions, one node.

In [ ]:
b = BDD(3)
x, y, z = b.var(0), b.var(1), b.var(2)
f1 = b.AND(x, b.OR(y, z))
f2 = b.OR(b.AND(x, y), b.AND(x, z))            # distributive law
print("f1 is f2 ?", f1 is f2)
assert f1 is f2
print("\nNo comparison was performed.  They are the SAME OBJECT.")

Equivalence checking is therefore $O(1)$.

In [ ]:
import time
N = 14
b = BDD(N)
xs = [b.var(i) for i in range(N)]
a1 = xs[0]
for v in xs[1:]: a1 = b.AND(a1, v)
a2 = xs[-1]
for v in reversed(xs[:-1]): a2 = b.AND(v, a2)
t0 = time.time(); same = equivalent(b, a1, a2); t1 = time.time()
print("AND built left-to-right vs right-to-left, %d vars" % N)
print("   equivalent? %s  in %.7f seconds" % (same, t1 - t0))
assert same

**Tautology and unsatisfiability** are terminal-node tests.

In [ ]:
taut = b.OR(xs[0], b.NOT(xs[0]))
unsat = b.AND(xs[0], b.NOT(xs[0]))
print("x OR NOT x is a tautology ? ", is_tautology(b, taut))
print("x AND NOT x is unsat      ? ", is_unsat(b, unsat))
assert is_tautology(b, taut) and is_unsat(b, unsat)
print("\nCompare Chapter 16: SAT is NP-complete.  The hard work moved into")
print("BUILDING the BDD -- which can blow up.  Querying it is free.")

**Apply** with memoisation, and the cache doing its job.

In [ ]:
b2 = BDD(8)
ys = [b2.var(i) for i in range(8)]
before = len(b2.apply_cache)
g = ys[0]
for v in ys[1:]: g = b2.XOR(g, v)
print("parity over 8 vars : %d nodes, %d apply-cache entries"
      % (b2.size(g), len(b2.apply_cache) - before))
print("without the cache, apply would re-derive shared subgraphs repeatedly")

De Morgan and double negation, verified by pointer equality.

In [ ]:
b3 = BDD(3)
p, q = b3.var(0), b3.var(1)
assert b3.NOT(b3.NOT(p)) is p
assert b3.NOT(b3.AND(p, q)) is b3.OR(b3.NOT(p), b3.NOT(q))
assert b3.NOT(b3.OR(p, q)) is b3.AND(b3.NOT(p), b3.NOT(q))
print("NOT NOT p          is p            : verified by identity")
print("NOT (p AND q)      is NOT p OR NOT q")
print("NOT (p OR q)       is NOT p AND NOT q")
print("\nThree laws of Boolean algebra, checked with `is`.")

The chain of ideas, in one line.

In [ ]:
print("Myhill-Nerode  ->  canonical minimal DFA")
print("               ->  canonical reduced BDD (fixed variable order)")
print("hash consing   ->  canonical means IDENTICAL, not merely isomorphic")
print("               ->  equivalence checking is a pointer comparison")

## 4. Exercises


1. Why does canonicity require a **fixed** variable order?
2. What is the worst-case cost of `apply` without the cache?
3. How would you check implication $f \Rightarrow g$ with BDDs?

In [ ]:
# Your work for the exercises above.